# 06 — Live Inference

Run the trained YOLOv11 checkpoint on:

| Section | Input |
|---------|-------|
| 3 | Single image (file path) |
| 4 | Folder / batch of images |
| 5 | Video file |
| 6 | Webcam live feed (stop button) |

In [1]:
# Install dependencies
%pip install -q "ultralytics==8.3.*" pillow ipywidgets
print("nb06 dependencies ready")

Note: you may need to restart the kernel to use updated packages.
nb06 dependencies ready


In [2]:
%matplotlib inline

import io
import random
import time
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as IPImage
from IPython.display import clear_output, display
from PIL import Image
from ultralytics import YOLO

In [3]:
# ── Config ────────────────────────────────────────────────────────────────────
WEIGHTS      = Path('../weights/best.pt').resolve()
TEST_IMG_DIR = Path('../data/dataset/images/test')
CLASSES      = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']

IMGSZ       = 640
CONF_THRESH = 0.25
SEED        = 42

assert WEIGHTS.exists(), f'Weights not found: {WEIGHTS} — run notebook 04 first'
random.seed(SEED)
print(f'Weights : {WEIGHTS}')
print(f'Classes : {CLASSES}')

Weights : C:\Users\mitah\github_projects\ai_cv_project\weights\best.pt
Classes : ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']


In [4]:
# Load model and warm up so first-inference latency doesn't skew timing
model = YOLO(WEIGHTS)
_ = model.predict(np.zeros((640, 640, 3), dtype=np.uint8), imgsz=IMGSZ, verbose=False)
print(f'Model loaded — {sum(p.numel() for p in model.model.parameters())/1e6:.1f}M params')
print('Warm-up done — ready for inference')

Model loaded — 2.6M params
Warm-up done — ready for inference


## 1. Single image

In [5]:
# Change IMAGE_PATH to any image you want to test.
# Defaults to a random test-set image if left as None.
IMAGE_PATH = "data\dataset\images\test\0682_door_sign.jpg"  # e.g. Path('my_photo.jpg') or r'C:\path\to\image.jpg'

if IMAGE_PATH is None:
    imgs = list(TEST_IMG_DIR.glob('*.jpg'))
    assert imgs, f'No images found in {TEST_IMG_DIR}'
    IMAGE_PATH = random.choice(imgs)

t0 = time.perf_counter()
result = model.predict(str(IMAGE_PATH), imgsz=IMGSZ, conf=CONF_THRESH, verbose=False)[0]
latency_ms = (time.perf_counter() - t0) * 1000

# ── Print detections ─────────────────────────────────────────────────────────
print(f'Image   : {Path(IMAGE_PATH).name}')
print(f'Latency : {latency_ms:.1f} ms')
print(f'Detected: {len(result.boxes)} object(s)')
print()
if len(result.boxes):
    print(f"  {'class':<22} {'conf':>6}  {'xyxy box'}")
    print('  ' + '-' * 60)
    for box in result.boxes:
        cls_name = CLASSES[int(box.cls)]
        conf     = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        print(f"  {cls_name:<22} {conf:>6.3f}  [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")
else:
    print('  (no detections above confidence threshold)')

# ── Display annotated image ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(Image.open(IMAGE_PATH))
axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(result.plot()[:, :, ::-1])
axes[1].set_title(f'Predictions ({len(result.boxes)} detections)'); axes[1].axis('off')
plt.tight_layout()
plt.show()

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\mitah\AppData\Local\Temp\ipykernel_16892\64404565.py:3: SyntaxWarning: invalid escape sequence '\d'
  IMAGE_PATH = "data\dataset\images\test\0682_door_sign.jpg"  # e.g. Path('my_photo.jpg') or r'C:\path\to\image.jpg'


FileNotFoundError: data\dataset\images	est82_door_sign.jpg does not exist

## 2. Batch / folder inference

In [ ]:
# Set FOLDER to any directory of images, or leave None to use the test split.
FOLDER    = None   # e.g. Path('my_images/')
N_SHOW    = 8      # max images to show in the grid
GRID_COLS = 4

if FOLDER is None:
    FOLDER = TEST_IMG_DIR

all_imgs = sorted(Path(FOLDER).glob('*.jpg')) + sorted(Path(FOLDER).glob('*.png'))
assert all_imgs, f'No jpg/png images found in {FOLDER}'

t0 = time.perf_counter()
results = model.predict(
    source=[str(p) for p in all_imgs],
    imgsz=IMGSZ, conf=CONF_THRESH, verbose=False
)
elapsed = (time.perf_counter() - t0) * 1000

total_dets = sum(len(r.boxes) for r in results)
print(f'Images   : {len(results)}')
print(f'Detections: {total_dets} total  ({total_dets/len(results):.1f} avg per image)')
print(f'Throughput: {len(results)/(elapsed/1000):.1f} imgs/s  ({elapsed/len(results):.1f} ms/img)')

# ── Per-class detection counts ────────────────────────────────────────────────
counts = {c: 0 for c in CLASSES}
for r in results:
    for box in r.boxes:
        counts[CLASSES[int(box.cls)]] += 1
print()
print(f"  {'class':<22} {'count':>6}")
print('  ' + '-' * 30)
for cls, cnt in counts.items():
    print(f"  {cls:<22} {cnt:>6}")

# ── Grid of annotated images ──────────────────────────────────────────────────
sample_results = random.sample(results, min(N_SHOW, len(results)))
rows = (len(sample_results) + GRID_COLS - 1) // GRID_COLS
fig, axes = plt.subplots(rows, GRID_COLS, figsize=(4 * GRID_COLS, 4 * rows))
axes = np.array(axes).flatten()
for ax, r in zip(axes, sample_results):
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(f'{Path(r.path).name}\n{len(r.boxes)} det', fontsize=7)
    ax.axis('off')
for ax in axes[len(sample_results):]:
    ax.axis('off')
plt.suptitle(f'Batch inference — {len(sample_results)} of {len(results)} images shown', y=1.01)
plt.tight_layout()
plt.show()

## 3. Video file inference

In [ ]:
# Set VIDEO_PATH to your video file.
VIDEO_PATH  = None   # e.g. Path('campus_clip.mp4')
N_PREVIEW   = 6      # number of evenly-spaced frames to show as a preview grid

if VIDEO_PATH is None:
    print('VIDEO_PATH is None — skipping video inference.')
    print('Set VIDEO_PATH to a .mp4 / .avi file and re-run.')
else:
    VIDEO_PATH = Path(VIDEO_PATH)
    assert VIDEO_PATH.exists(), f'File not found: {VIDEO_PATH}'

    out_path = VIDEO_PATH.parent / f'{VIDEO_PATH.stem}_annotated{VIDEO_PATH.suffix}'

    cap = cv2.VideoCapture(str(VIDEO_PATH))
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    preview_frames, frame_idx, det_total = [], 0, 0
    preview_at = set(range(0, total, max(1, total // N_PREVIEW)))  # evenly spaced indices
    t0 = time.perf_counter()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        result    = model.predict(frame, imgsz=IMGSZ, conf=CONF_THRESH, verbose=False)[0]
        annotated = result.plot()
        writer.write(annotated)
        det_total += len(result.boxes)
        if frame_idx in preview_at:
            preview_frames.append((frame_idx, annotated[:, :, ::-1].copy()))
        frame_idx += 1

    cap.release()
    writer.release()
    elapsed = time.perf_counter() - t0

    print(f'Frames     : {frame_idx}')
    print(f'Detections : {det_total} total  ({det_total/frame_idx:.2f} avg/frame)')
    print(f'Speed      : {frame_idx/elapsed:.1f} fps processed  ({elapsed/frame_idx*1000:.1f} ms/frame)')
    print(f'Saved to   : {out_path}')

    # ── Preview grid of evenly-spaced annotated frames ────────────────────────
    cols = min(3, len(preview_frames))
    rows = (len(preview_frames) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).flatten()
    for ax, (fidx, img) in zip(axes, preview_frames):
        ax.imshow(img)
        ax.set_title(f'frame {fidx}', fontsize=8)
        ax.axis('off')
    for ax in axes[len(preview_frames):]:
        ax.axis('off')
    plt.suptitle(f'Video preview — {VIDEO_PATH.name}', y=1.01)
    plt.tight_layout()
    plt.show()

## 4. Webcam live feed

Streams from the default webcam (index `0`). Click **Stop** or interrupt the kernel to end the loop.

In [ ]:
WEBCAM_INDEX = 0      # change if you have multiple cameras
RUN_WEBCAM   = False  # set True to start the live feed

if not RUN_WEBCAM:
    print('RUN_WEBCAM is False — skipping.')
    print('Set RUN_WEBCAM = True and re-run this cell to start the live feed.')
else:
    stop_btn = widgets.Button(description='Stop', button_style='danger')
    out      = widgets.Output()
    display(stop_btn, out)

    running = [True]
    stop_btn.on_click(lambda _: running.__setitem__(0, False))

    cap = cv2.VideoCapture(WEBCAM_INDEX)
    assert cap.isOpened(), f'Cannot open camera index {WEBCAM_INDEX}'

    frame_count, det_total, t_start = 0, 0, time.perf_counter()
    try:
        while running[0]:
            ret, frame = cap.read()
            if not ret:
                print('Camera read failed — stopping.')
                break

            result    = model.predict(frame, imgsz=IMGSZ, conf=CONF_THRESH, verbose=False)[0]
            annotated = result.plot()
            det_total += len(result.boxes)
            frame_count += 1

            fps_live = frame_count / max(time.perf_counter() - t_start, 1e-9)

            # Encode to JPEG and push to the output widget
            _, buf = cv2.imencode('.jpg', annotated, [cv2.IMWRITE_JPEG_QUALITY, 85])
            with out:
                clear_output(wait=True)
                display(IPImage(data=buf.tobytes()))
                print(f'Frame {frame_count:>5}  |  {fps_live:>5.1f} fps  |  '
                      f'{len(result.boxes)} det this frame  |  '
                      f'{det_total} total')
    finally:
        cap.release()
        elapsed = time.perf_counter() - t_start
        print(f'\nStopped after {frame_count} frames ({elapsed:.1f}s, '
              f'{frame_count/elapsed:.1f} fps avg, {det_total} total detections)')